# 13 — Interview Q&A: Security, Databases, Testing & Advanced

MCQs and detailed Q&A covering:
- Error Handling
- Security & Performance
- Databases & ORMs
- Testing & Deployment
- Advanced Topics (Worker Threads, Design Patterns, Microservices)

---

# PART A — Multiple Choice Questions

---

## Section 1: Error Handling MCQs

### MCQ 1
**What is the difference between operational errors and programmer errors?**

A) Operational errors are bugs; programmer errors are expected failures  
B) Operational errors are expected runtime failures; programmer errors are bugs in code  
C) Both are the same, just different naming conventions  
D) Operational errors only occur in production; programmer errors only in development  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Operational errors are expected runtime failures; programmer errors are bugs in code**

Operational errors (network timeout, invalid user input, file not found) are expected and should be handled gracefully. Programmer errors (TypeError, null reference, wrong argument types) are bugs — the process is in an unknown state and should be restarted.
</details>

### MCQ 2
**What should you do after catching an `uncaughtException`?**

A) Log it and continue running normally  
B) Retry the failed operation  
C) Log it, close connections gracefully, and exit the process  
D) Ignore it — the process will recover on its own  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) Log it, close connections gracefully, and exit the process**

After an uncaught exception, the process state is potentially corrupted. Continuing to handle requests could lead to data corruption, memory leaks, or security vulnerabilities. Log the error, close DB connections and HTTP server, then `process.exit(1)`. Use PM2 or Docker to automatically restart.
</details>

### MCQ 3
**What does `Error.captureStackTrace(this, this.constructor)` do in a custom error class?**

A) Captures the current call stack, excluding the constructor from the trace  
B) Logs the stack trace to the console  
C) Sends the stack trace to an error monitoring service  
D) Prevents the error from being caught  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Captures the current call stack, excluding the constructor from the trace**

This V8-specific method creates a `.stack` property on the error object. The second argument tells V8 where to stop the trace — passing `this.constructor` excludes the error class constructor itself, so the trace starts from the code that threw the error, making it cleaner.
</details>

---
## Section 2: Security MCQs

### MCQ 4
**Which is the most effective way to prevent SQL injection?**

A) Escaping special characters in user input  
B) Using parameterized/prepared queries  
C) Limiting input string length  
D) Using POST instead of GET requests  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Using parameterized/prepared queries**

Parameterized queries separate SQL code from data — the database engine treats user input as a literal value, never as executable SQL. Escaping (A) is error-prone and bypassable. Input length limits (C) help but don't solve the root cause. POST vs GET (D) is irrelevant to injection.

```javascript
// SAFE: parameterized query
db.query('SELECT * FROM users WHERE email = $1', [userInput]);
```
</details>

### MCQ 5
**What is the structure of a JWT (JSON Web Token)?**

A) `USERNAME.PASSWORD.TIMESTAMP`  
B) `HEADER.PAYLOAD.SIGNATURE`  
C) `TOKEN.REFRESH.EXPIRY`  
D) `KEY.VALUE.HASH`  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) `HEADER.PAYLOAD.SIGNATURE`**

- **Header** — algorithm and token type: `{ "alg": "HS256", "typ": "JWT" }`
- **Payload** — claims (data): `{ "sub": "user123", "role": "admin", "exp": 1700000000 }`
- **Signature** — HMAC of header + payload using a secret key

Each part is Base64URL-encoded and separated by dots. The signature ensures the token hasn't been tampered with.
</details>

### MCQ 6
**What does the `helmet` middleware do?**

A) Encrypts all request/response data  
B) Sets various HTTP security headers to protect against common attacks  
C) Adds authentication to all routes  
D) Rate limits incoming requests  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Sets various HTTP security headers to protect against common attacks**

Helmet sets ~15 security headers including: `X-Content-Type-Options: nosniff`, `X-Frame-Options: DENY` (clickjacking), `Strict-Transport-Security` (force HTTPS), Content-Security-Policy, and removes `X-Powered-By: Express` (hides server identity). One line: `app.use(helmet())`.
</details>

### MCQ 7
**What is CORS?**

A) A Node.js framework for building APIs  
B) A browser security mechanism that restricts cross-origin HTTP requests  
C) A database caching strategy  
D) A testing methodology for REST APIs  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) A browser security mechanism that restricts cross-origin HTTP requests**

CORS (Cross-Origin Resource Sharing) prevents a web page from making requests to a different origin (domain, port, or protocol) unless the server explicitly allows it via response headers. The server must set `Access-Control-Allow-Origin` and related headers. In Express, use the `cors` package.
</details>

### MCQ 8
**Which hashing algorithm should you use for passwords?**

A) MD5  
B) SHA-256  
C) bcrypt  
D) Base64  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) bcrypt**

bcrypt is specifically designed for password hashing — it's deliberately slow (configurable via salt rounds) to resist brute-force attacks. MD5 and SHA-256 are too fast (designed for data integrity, not passwords). Base64 is encoding, not hashing at all. Alternatives to bcrypt: scrypt, argon2.
</details>

---
## Section 3: Performance & Scaling MCQs

### MCQ 9
**What does the `cluster` module do?**

A) Clusters database connections  
B) Forks multiple Node.js worker processes that share the same server port  
C) Groups related routes together  
D) Manages npm package dependencies  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Forks multiple Node.js worker processes that share the same server port**

The cluster module creates multiple copies of your Node.js process (one per CPU core). A primary process manages the workers, and the OS distributes incoming connections among them. This lets a Node.js server use all CPU cores for handling requests.
</details>

### MCQ 10
**What is the default size of the libuv thread pool?**

A) 1  
B) 4  
C) 8  
D) Equal to the number of CPU cores  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) 4**

libuv's thread pool defaults to 4 threads. It can be changed via the `UV_THREADPOOL_SIZE` environment variable (max 1024). The thread pool is used for blocking operations that can't use OS async primitives: file system I/O, DNS lookups (`dns.lookup()`), crypto operations, and zlib compression.
</details>

### MCQ 11
**Which is NOT a common cause of memory leaks in Node.js?**

A) Global variables that grow indefinitely  
B) Event listeners added on every request but never removed  
C) Using `const` to declare variables  
D) Closures that hold references to large objects  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) Using `const` to declare variables**

`const` is a best practice and does NOT cause memory leaks. Common leak sources: global caches without eviction (A), event listeners added per-request without cleanup (B), closures keeping references to large data (D), forgotten timers (`setInterval` without `clearInterval`), and circular references in data structures.
</details>

---
## Section 4: Database MCQs

### MCQ 12
**What is the N+1 query problem?**

A) A query that takes N+1 seconds to execute  
B) Fetching N records then making 1 additional query per record for related data  
C) A database that can only handle N+1 connections  
D) An indexing strategy for N+1 columns  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Fetching N records then making 1 additional query per record for related data**

Example: Fetch 100 users (1 query), then for each user, fetch their posts (100 queries) = 101 queries total. Fix: use eager loading — `User.find().populate('posts')` (Mongoose), `User.findAll({ include: Post })` (Sequelize), or JOIN in SQL.
</details>

### MCQ 13
**What does ACID stand for in database transactions?**

A) Accessible, Concurrent, Integrated, Distributed  
B) Atomicity, Consistency, Isolation, Durability  
C) Asynchronous, Cached, Indexed, Deduplicated  
D) Automated, Computed, Incremental, Dynamic  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Atomicity, Consistency, Isolation, Durability**

- **Atomicity** — All operations succeed or all fail (no partial updates)
- **Consistency** — Data moves from one valid state to another
- **Isolation** — Concurrent transactions don't interfere with each other
- **Durability** — Committed data survives system crashes
</details>

### MCQ 14
**What is connection pooling?**

A) Connecting to multiple databases simultaneously  
B) Maintaining a set of reusable database connections to avoid per-request connection overhead  
C) Encrypting database connections  
D) Load balancing across database replicas  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Maintaining a set of reusable database connections to avoid per-request connection overhead**

Creating a new DB connection takes 20-50ms. A pool maintains open connections that are borrowed for queries and returned when done. This eliminates the overhead of creating/destroying connections per request. Typical pool sizes: 5-20 connections.
</details>

### MCQ 15
**In MongoDB, when should you embed a document vs reference it?**

A) Always embed for better performance  
B) Always reference to keep data normalized  
C) Embed when data is always accessed together and is bounded; reference when data is accessed independently or can grow unbounded  
D) Embed for large data, reference for small data  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) Embed when data is always accessed together and is bounded; reference when data is accessed independently or can grow unbounded**

Embed: User's address (1:1, always fetched with user), order line items (bounded, accessed with order). Reference: User's posts (could be thousands, often paginated independently), many-to-many relationships. MongoDB documents have a 16MB size limit, which is another reason to reference for unbounded data.
</details>

---
## Section 5: Testing MCQs

### MCQ 16
**What is the testing pyramid (from most to fewest tests)?**

A) E2E → Integration → Unit  
B) Unit → Integration → E2E  
C) Integration → Unit → E2E  
D) All types should have equal numbers  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Unit → Integration → E2E**

The pyramid means: MOST unit tests (fast, cheap, test individual functions), SOME integration tests (test components working together), FEW E2E tests (slow, expensive, test full user workflows). This gives the best balance of coverage, speed, and maintenance cost.
</details>

### MCQ 17
**What is the difference between `toBe()` and `toEqual()` in Jest?**

A) No difference  
B) `toBe` uses strict equality (===), `toEqual` uses deep equality for objects  
C) `toBe` is for numbers, `toEqual` is for strings  
D) `toBe` checks type, `toEqual` checks value  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) `toBe` uses strict equality (===), `toEqual` uses deep equality for objects**

```javascript
expect(1 + 1).toBe(2);                // passes (primitive comparison)
expect({ a: 1 }).toBe({ a: 1 });      // FAILS (different object references)
expect({ a: 1 }).toEqual({ a: 1 });   // passes (deep comparison)
```

Use `toBe` for primitives and reference checks. Use `toEqual` for objects and arrays.
</details>

### MCQ 18
**What does Supertest do?**

A) Tests CPU performance of Node.js applications  
B) Makes HTTP assertions against an Express app without starting a server  
C) Tests database queries for performance  
D) Runs browser-based E2E tests  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Makes HTTP assertions against an Express app without starting a server**

Supertest wraps your Express app, sends HTTP requests to it, and lets you assert on status codes, headers, and response bodies — all without calling `app.listen()`. This makes API testing fast and doesn't require an open port.

```javascript
const request = require('supertest');
await request(app).get('/users').expect(200).expect('Content-Type', /json/);
```
</details>

---
## Section 6: Advanced Topics MCQs

### MCQ 19
**What is the key difference between Worker Threads and Child Processes?**

A) Worker Threads are slower  
B) Worker Threads share memory (SharedArrayBuffer), Child Processes have separate memory  
C) Child Processes can only run JavaScript  
D) Worker Threads require a separate machine  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Worker Threads share memory (SharedArrayBuffer), Child Processes have separate memory**

Worker Threads share the same process and can share memory via `SharedArrayBuffer` (lower overhead). Child Processes are completely separate OS processes with isolated memory (higher overhead, better isolation). Use Worker Threads for CPU-intensive JS tasks, Child Processes for running external programs or when you need full isolation.
</details>

### MCQ 20
**Which design pattern does Node.js module caching naturally implement?**

A) Factory  
B) Observer  
C) Singleton  
D) Strategy  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) Singleton**

Since `require()` caches the module after the first call, `module.exports = new MyService()` creates a singleton — every file that requires this module gets the exact same instance. This is the easiest way to implement singletons in Node.js.
</details>

### MCQ 21
**What is the Circuit Breaker pattern?**

A) A way to encrypt network traffic  
B) A pattern that prevents cascading failures by failing fast after repeated errors  
C) A database sharding strategy  
D) A method of load balancing requests  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) A pattern that prevents cascading failures by failing fast after repeated errors**

Three states: **CLOSED** (normal operation), **OPEN** (after failure threshold — returns errors immediately without calling the service), **HALF-OPEN** (after timeout — allows one test request). Used in microservices to prevent a failing service from taking down the entire system.
</details>

### MCQ 22
**Which child process method provides built-in IPC (Inter-Process Communication)?**

A) `exec()`  
B) `spawn()`  
C) `fork()`  
D) `execFile()`  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: C) `fork()`**

`fork()` is a special form of `spawn()` specifically for Node.js scripts. It creates a new V8 instance with a built-in IPC channel — you can use `child.send(message)` and `child.on('message', handler)` for parent-child communication. `exec()` and `spawn()` don't have built-in IPC.
</details>

### MCQ 23
**What is the main advantage of GraphQL over REST?**

A) GraphQL is always faster  
B) GraphQL prevents over-fetching and under-fetching by letting clients request exact data  
C) GraphQL doesn't need a server  
D) GraphQL automatically caches all responses  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) GraphQL prevents over-fetching and under-fetching by letting clients request exact data**

In REST, the server defines the response shape. In GraphQL, the client specifies exactly which fields it needs. This eliminates over-fetching (getting unnecessary data) and under-fetching (needing multiple requests for related data). However, REST has simpler caching, and GraphQL is NOT inherently faster — it depends on the use case.
</details>

### MCQ 24
**In a Dockerfile for Node.js, why should you use `npm ci` instead of `npm install`?**

A) `npm ci` is newer  
B) `npm ci` provides reproducible builds by using `package-lock.json` exactly  
C) `npm ci` installs fewer packages  
D) `npm ci` doesn't require a `package.json`  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) `npm ci` provides reproducible builds by using `package-lock.json` exactly**

`npm ci` deletes `node_modules` and installs the exact versions from `package-lock.json` without modifying it. This ensures the Docker image has the exact same dependencies every time. `npm install` may resolve slightly different versions and can update the lock file.
</details>

### MCQ 25
**What is PM2 primarily used for?**

A) Package management  
B) Process management — running Node.js apps in production with clustering, auto-restart, and monitoring  
C) Database migrations  
D) Testing automation  

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Process management — running Node.js apps in production with clustering, auto-restart, and monitoring**

PM2 handles: cluster mode (use all CPU cores), automatic restart on crash, zero-downtime reloads, log management, memory limit restart, startup scripts, and a monitoring dashboard. Key commands: `pm2 start app.js -i max`, `pm2 reload app`, `pm2 monit`.
</details>

---
## Section 7: Additional Security, DB, Testing & Advanced MCQs

### MCQ 26
**What is the difference between `process.on('unhandledRejection')` behavior in Node.js 14 vs 15+?**

A) Both versions log a warning and continue running  
B) In Node.js 15+, unhandled rejections throw and crash the process by default; in 14 and earlier they only logged a deprecation warning  
C) Both versions crash the process immediately  
D) Node.js 14 crashes; Node.js 15+ only logs

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) In Node.js 15+, unhandled rejections throw and crash the process by default; in 14 and earlier they only logged a deprecation warning**

Node.js 15+ changed the default behavior to terminate the process on unhandled promise rejections, treating them like uncaught exceptions. In Node 14 and earlier, unhandled rejections only emitted a deprecation warning. You can restore the old behavior with `--unhandled-rejections=warn`.
</details>

### MCQ 27
**What is XSS (Cross-Site Scripting) and how do you prevent it?**

A) A database injection attack; prevent with parameterized queries  
B) Injecting malicious scripts into web pages; prevent with output encoding, Content-Security-Policy headers, and sanitizing HTML input  
C) Stealing session cookies; prevent with httpOnly cookies only  
D) A type of CSRF attack; prevent with CSRF tokens

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) Injecting malicious scripts into web pages; prevent with output encoding, Content-Security-Policy headers, and sanitizing HTML input**

XSS occurs when an attacker injects malicious JavaScript into a page that is then executed in other users' browsers. Prevention: encode output (escape `<`, `>`, `"`, `'`), use Content-Security-Policy to restrict script sources, sanitize HTML with libraries like DOMPurify, and use httpOnly cookies to limit cookie theft impact.
</details>

### MCQ 28
**What is a database index and when should you NOT create one?**

A) An index speeds up reads but slows writes; don't create on rarely queried fields, low-cardinality fields (boolean), small tables, or write-heavy tables  
B) An index speeds up both reads and writes; create on every column  
C) An index is only for primary keys; never create on other columns  
D) An index is for caching; don't create on frequently updated data

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) An index speeds up reads but slows writes; don't create on rarely queried fields, low-cardinality fields (boolean), small tables, or write-heavy tables**

Indexes speed up SELECT queries but add overhead to INSERT, UPDATE, and DELETE. Avoid indexes on: columns rarely used in WHERE/JOIN, low-cardinality columns (few distinct values like boolean), small tables (full scan may be faster), and tables with heavy write load where the index maintenance cost outweighs read benefits.
</details>

### MCQ 29
**What is the difference between `jest.fn()`, `jest.mock()`, and `jest.spyOn()`?**

A) They are interchangeable; all create mocks  
B) `fn()` creates a mock function; `mock()` replaces an entire module; `spyOn()` wraps an existing method to track calls while keeping original behavior  
C) `fn()` is for unit tests; `mock()` is for integration; `spyOn()` is for E2E  
D) `spyOn()` replaces the module; `mock()` creates a function; `fn()` wraps a method

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: B) `fn()` creates a mock function; `mock()` replaces an entire module; `spyOn()` wraps an existing method to track calls while keeping original behavior**

- **jest.fn()** — Creates a mock function you can assert on: `const mockFn = jest.fn(); mockFn('arg'); expect(mockFn).toHaveBeenCalledWith('arg');`
- **jest.mock()** — Replaces an entire module with a mock: `jest.mock('./mailer', () => ({ send: jest.fn() }));`
- **jest.spyOn()** — Wraps a real method to track calls while optionally keeping original: `jest.spyOn(obj, 'method').mockImplementation(() => 'mocked');`
</details>

### MCQ 30
**What is the Saga pattern in microservices?**

A) A pattern for managing distributed transactions across services by breaking them into a sequence of local transactions with compensating actions on failure  
B) A database replication strategy  
C) A load balancing algorithm  
D) A testing pattern for integration tests

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) A pattern for managing distributed transactions across services by breaking them into a sequence of local transactions with compensating actions on failure**

In a Saga, each service performs a local transaction and publishes an event. If a step fails, compensating transactions (undo operations) are executed in reverse order to roll back. Example: Order service creates order → Payment service charges → Inventory service reserves; if inventory fails, run compensating: refund payment, cancel order.
</details>

### MCQ 31
**What is the difference between authentication and authorization?**

A) Authentication = verifying identity (who are you?); Authorization = verifying permissions (what can you do?)  
B) Authentication = verifying permissions; Authorization = verifying identity  
C) They are the same concept with different names  
D) Authentication is for API keys; Authorization is for JWT tokens

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Authentication = verifying identity (who are you?); Authorization = verifying permissions (what can you do?)**

**Authentication** — Confirms the user is who they claim to be (login, JWT verification, API key validation).  
**Authorization** — Determines what an authenticated user is allowed to do (role checks, permission checks, resource ownership). Example: User logs in (auth) → User tries to delete another user's post (authorization denies).
</details>

### MCQ 32
**What is database sharding?**

A) Distributing data across multiple database instances based on a shard key to handle more data/traffic than a single server can  
B) Splitting a table into multiple columns  
C) A caching strategy for frequently accessed data  
D) A backup strategy that copies data to multiple locations

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Distributing data across multiple database instances based on a shard key to handle more data/traffic than a single server can**

Sharding horizontally partitions data: each shard holds a subset (e.g., users A–M on shard 1, N–Z on shard 2). The shard key (e.g., user_id, region) determines which shard stores/retrieves the data. Used when a single DB can't handle the load. Challenges: cross-shard queries, rebalancing, choosing a good shard key.
</details>

### MCQ 33
**What is the purpose of a reverse proxy (like Nginx) in front of Node.js?**

A) SSL termination, static file serving, load balancing, request buffering, rate limiting, caching, and hiding internal architecture  
B) Only for load balancing across Node.js instances  
C) To connect Node.js to the database  
D) To handle authentication before requests reach Node.js

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) SSL termination, static file serving, load balancing, request buffering, rate limiting, caching, and hiding internal architecture**

A reverse proxy sits between clients and your Node.js app. It handles: **SSL/TLS termination** (decrypt HTTPS, forward HTTP to Node), **static files** (serve CSS/JS/images without hitting Node), **load balancing** (distribute to multiple Node instances), **request buffering** (protects Node from slow clients), **rate limiting**, **caching**, and **hiding** internal structure (clients don't see Node directly).
</details>

### MCQ 34
**What does the `--max-old-space-size` flag do?**

A) Sets the maximum V8 heap size in MB (default ~1.5GB); use when processing large datasets that need more memory  
B) Sets the maximum number of CPU cores Node.js can use  
C) Sets the maximum number of file descriptors  
D) Sets the maximum depth of the event loop

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Sets the maximum V8 heap size in MB (default ~1.5GB); use when processing large datasets that need more memory**

Example: `node --max-old-space-size=4096 app.js` sets the heap limit to 4GB. Use when your app hits "JavaScript heap out of memory" errors while processing large JSON, big in-memory caches, or heavy data transformations. Default is ~1.5GB on 64-bit systems.
</details>

### MCQ 35
**What is the difference between `docker build` multi-stage and single-stage for Node.js?**

A) Multi-stage uses a builder stage for npm install and copies only production files to a smaller final image, reducing size and attack surface  
B) Multi-stage builds are faster than single-stage  
C) Multi-stage is only for development; single-stage is for production  
D) They produce identical images; multi-stage is just a different syntax

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Multi-stage uses a builder stage for npm install and copies only production files to a smaller final image, reducing size and attack surface**

Single-stage: `FROM node`, `npm install`, `COPY .` — includes devDependencies, source code, and build tools in the final image. Multi-stage: first stage runs `npm ci` and builds; second stage `FROM node:alpine` copies only `node_modules` (production) and `dist/` — smaller image, no dev deps, fewer vulnerabilities.
</details>

### MCQ 36
**What is CSRF and how do you prevent it?**

A) Cross-Site Request Forgery tricks a user's browser into making unwanted requests; prevent with CSRF tokens, SameSite cookies, and checking the Origin header  
B) Cross-Site Scripting; prevent with output encoding  
C) A session fixation attack; prevent with secure session IDs  
D) SQL injection via cookies; prevent with parameterized queries

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Cross-Site Request Forgery tricks a user's browser into making unwanted requests; prevent with CSRF tokens, SameSite cookies, and checking the Origin header**

CSRF: A malicious site causes the user's browser to send a request to your site (with cookies) without the user's intent. Prevention: **CSRF tokens** (random value in form/session, validated on submit), **SameSite=Strict/Lax** cookies (limits cross-site sends), **Origin/Referer header** checks (reject requests from unexpected origins).
</details>

### MCQ 37
**What is the difference between Mongoose `save()` and `findByIdAndUpdate()`?**

A) `save()` runs all middleware/validators and creates a full document instance; `findByIdAndUpdate()` is faster but skips middleware by default unless `runValidators: true` is set  
B) They are identical in behavior  
C) `save()` is faster; `findByIdAndUpdate()` is for complex updates  
D) `findByIdAndUpdate()` always runs all middleware and validators

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) `save()` runs all middleware/validators and creates a full document instance; `findByIdAndUpdate()` is faster but skips middleware by default unless `runValidators: true` is set**

`save()` — Load document, modify in memory, call `doc.save()`. Runs pre/post save hooks, validators, and returns the full document.  
`findByIdAndUpdate()` — Direct MongoDB update. Skips `save` middleware; add `runValidators: true` for validation. Faster for simple updates. Use `save()` when you need middleware (e.g., hashing passwords on change).
</details>

### MCQ 38
**What is a refresh token and why is it used with JWT?**

A) A long-lived token used to obtain new short-lived access tokens without re-authentication; improves security by keeping access tokens short-lived while avoiding frequent logins  
B) A token that refreshes the user's session in the database  
C) An encrypted version of the JWT  
D) A token used to refresh database connections

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) A long-lived token used to obtain new short-lived access tokens without re-authentication; improves security by keeping access tokens short-lived while avoiding frequent logins**

Access tokens are short-lived (e.g., 15 min) to limit exposure if stolen. Refresh tokens are long-lived (days/weeks), stored securely (httpOnly cookie), and used to call `/refresh` and get a new access token. If a refresh token is stolen, it can be revoked server-side. This balances security (short access token) with UX (no constant re-login).
</details>

### MCQ 39
**What is the difference between integration tests and E2E tests?**

A) Integration tests verify components working together (API + DB); E2E tests simulate real user workflows through the entire application (browser → API → DB → response)  
B) They are the same; both test the full stack  
C) Integration tests use a browser; E2E tests don't  
D) E2E tests are faster and run more frequently

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Integration tests verify components working together (API + DB); E2E tests simulate real user workflows through the entire application (browser → API → DB → response)**

**Integration tests** — Test multiple units together (e.g., API route + real or in-memory DB). No browser; use Supertest. Faster, more focused.  
**E2E tests** — Test the full system as a user would: browser (Playwright/Cypress) → frontend → API → DB. Slower, catch integration issues across the whole stack. Both are valuable; use the testing pyramid.
</details>

### MCQ 40
**What is event-driven architecture vs request-driven architecture?**

A) Request-driven = client sends request, waits for response (REST); Event-driven = services emit events that other services react to asynchronously (Kafka, RabbitMQ)  
B) They are the same; both use HTTP  
C) Event-driven is always faster than request-driven  
D) Request-driven uses WebSockets; event-driven uses REST

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Request-driven = client sends request, waits for response (REST); Event-driven = services emit events that other services react to asynchronously (Kafka, RabbitMQ)**

**Request-driven (synchronous)** — Client calls API, waits for response. Tight coupling; client must know service locations.  
**Event-driven (asynchronous)** — Services publish events to a message broker; other services subscribe and react. Loose coupling, better for scaling and resilience. Example: Order service publishes "OrderCreated"; Notification and Inventory services consume it independently.
</details>

### MCQ 41
**What is the `Proxy` object in JavaScript and how is it used in Node.js?**

A) A Proxy wraps an object and intercepts operations like property access, assignment, and function calls; used in ORMs, validation layers, and reactive frameworks  
B) A network proxy for HTTP requests  
C) A database connection proxy  
D) An HTTP reverse proxy

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) A Proxy wraps an object and intercepts operations like property access, assignment, and function calls; used in ORMs, validation layers, and reactive frameworks**

`new Proxy(target, handler)` lets you intercept `get`, `set`, `has`, `deleteProperty`, etc. Use cases: ORMs (lazy loading on property access), validation (validate on set), reactive state (track changes), logging/mocking. Example: `const p = new Proxy(obj, { get(t, k) { console.log('get', k); return t[k]; } });`
</details>

### MCQ 42
**What is `AbortController` used for in Node.js?**

A) To cancel async operations like fetch requests, setTimeout, fs operations, and stream operations via an AbortSignal  
B) To abort the entire Node.js process  
C) To abort database transactions  
D) To abort HTTP requests only (not other async operations)

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) To cancel async operations like fetch requests, setTimeout, fs operations, and stream operations via an AbortSignal**

`AbortController` creates a signal you pass to cancellable APIs. Call `controller.abort()` to cancel. Supported in: `fetch(url, { signal })`, `fs.readFile(path, { signal })`, `events.once(emitter, 'event', { signal })`, `setTimeout` with `AbortSignal.timeout()`, and many stream APIs. Essential for request timeouts and cleanup.
</details>

### MCQ 43
**What is the difference between horizontal scaling and vertical scaling?**

A) Vertical = add more CPU/RAM to one machine; Horizontal = add more machines behind a load balancer; Node.js benefits from both (cluster for vertical, load balancer for horizontal)  
B) Horizontal = add more RAM; Vertical = add more machines  
C) They are the same; both add more resources  
D) Vertical = microservices; Horizontal = monolith

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Vertical = add more CPU/RAM to one machine; Horizontal = add more machines behind a load balancer; Node.js benefits from both (cluster for vertical, load balancer for horizontal)**

**Vertical scaling** — Bigger machine (more cores, more RAM). Use Node's `cluster` module to utilize multiple cores. Limited by hardware ceiling.  
**Horizontal scaling** — More machines. Add servers behind Nginx/ALB. Stateless design required. Scales further. In practice: scale vertically first (cluster mode), then horizontally when needed.
</details>

### MCQ 44
**What is serverless and when should you use it with Node.js?**

A) Running functions without managing servers (AWS Lambda, Vercel); best for event-driven tasks, variable traffic, APIs with unpredictable load; not ideal for long-running processes or WebSockets  
B) Running without a database  
C) Running without authentication  
D) Running without an HTTP server

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) Running functions without managing servers (AWS Lambda, Vercel); best for event-driven tasks, variable traffic, APIs with unpredictable load; not ideal for long-running processes or WebSockets**

Serverless runs your code in response to events (HTTP, S3, SQS) without managing servers. **Good for:** APIs with spiky traffic, cron jobs, webhooks, event processing. **Not ideal for:** Long-running tasks (timeout limits), persistent WebSockets, cold starts affecting latency, stateful applications. Node.js is well-supported (AWS Lambda, Vercel, Netlify).
</details>

### MCQ 45
**What is the purpose of health check endpoints?**

A) A `/health` or `/healthz` endpoint that returns 200 when the service is operational; used by load balancers, Kubernetes, and monitoring tools to detect and replace unhealthy instances  
B) An endpoint that checks database connectivity only  
C) An endpoint for user authentication  
D) An endpoint that returns application logs

<details>
<summary><b>Click to reveal answer</b></summary>

**Answer: A) A `/health` or `/healthz` endpoint that returns 200 when the service is operational; used by load balancers, Kubernetes, and monitoring tools to detect and replace unhealthy instances**

Health endpoints let orchestrators know if the app is ready for traffic. **Liveness** — Is the process running? (simple 200). **Readiness** — Can it handle requests? (check DB, Redis, etc.). Kubernetes uses these for `livenessProbe` and `readinessProbe`. Load balancers stop sending traffic to instances that fail health checks.
</details>

---
# PART B — Detailed Interview Q&A

---

### Q1: How would you design a production-ready error handling strategy for a Node.js API?

**Model Answer:**

**Layer 1 — Custom Error Classes:**
```javascript
class AppError extends Error {
    constructor(message, statusCode) {
        super(message);
        this.statusCode = statusCode;
        this.isOperational = true;
        Error.captureStackTrace(this, this.constructor);
    }
}
class NotFoundError extends AppError { constructor(r) { super(`${r} not found`, 404); } }
class ValidationError extends AppError { constructor(m) { super(m, 400); } }
```

**Layer 2 — asyncHandler wrapper:**
```javascript
const asyncHandler = fn => (req, res, next) => Promise.resolve(fn(req, res, next)).catch(next);
```

**Layer 3 — Centralized error middleware (LAST in stack):**
```javascript
app.use((err, req, res, next) => {
    logger.error(err);
    const statusCode = err.statusCode || 500;
    const message = err.isOperational ? err.message : 'Internal server error';
    res.status(statusCode).json({ status: 'error', message });
});
```

**Layer 4 — Global safety nets:**
```javascript
process.on('uncaughtException', (err) => { logger.fatal(err); gracefulShutdown(1); });
process.on('unhandledRejection', (reason) => { throw reason; }); // Convert to uncaught
process.on('SIGTERM', () => gracefulShutdown(0));
```

**Key principles:** distinguish operational vs programmer errors, never expose internal details in production, always log with context (request ID, user ID), and exit on programmer errors.

### Q2: How would you secure a Node.js REST API in production?

**Model Answer (checklist):**

1. **HTTPS everywhere** — SSL/TLS termination at reverse proxy (Nginx)
2. **Helmet** — `app.use(helmet())` for security headers
3. **CORS** — Configure allowed origins, methods, headers
4. **Rate limiting** — `express-rate-limit` (global + stricter for auth routes)
5. **Input validation** — Joi/Zod on every endpoint
6. **Parameterized queries** — Never concatenate user input into SQL/MongoDB queries
7. **Password hashing** — bcrypt with 12+ salt rounds
8. **JWT best practices** — Short expiry (15min), refresh tokens, store in httpOnly cookies
9. **Environment variables** — Secrets in `.env` (never committed), validated at startup
10. **Dependencies** — `npm audit` regularly, update vulnerable packages
11. **Logging** — Log security events (failed logins, permission denials) without logging secrets
12. **Request size limits** — `express.json({ limit: '10kb' })` to prevent payload attacks
13. **HTTP Parameter Pollution** — Use `hpp` middleware
14. **NoSQL Injection prevention** — Sanitize MongoDB queries, reject `$` operators in user input
15. **Error messages** — Never expose stack traces or internal details in production

### Q3: How do you scale a Node.js application from 100 to 100,000 users?

**Model Answer:**

**Vertical scaling (single server):**
1. **Cluster mode** — PM2 with `pm2 start app.js -i max` (use all CPU cores)
2. **Caching** — Redis for sessions, frequently accessed data (reduce DB load)
3. **Database indexing** — Proper indexes on frequently queried fields
4. **Connection pooling** — Reuse DB connections (pool size tuned per server)
5. **Gzip compression** — `compression` middleware

**Horizontal scaling (multiple servers):**
6. **Load balancer** — Nginx or AWS ALB distributing traffic across servers
7. **Stateless design** — No in-process sessions (use Redis for shared state)
8. **Database read replicas** — Direct reads to replicas, writes to primary
9. **CDN** — Serve static assets from CloudFront/Cloudflare

**Architecture improvements:**
10. **Message queues** — Offload heavy tasks to background workers (BullMQ/RabbitMQ)
11. **Microservices** — Split monolith when team/features outgrow single codebase
12. **Database sharding** — Distribute data across multiple DB instances
13. **Auto-scaling** — AWS ECS/Kubernetes scales containers based on load

**Monitoring (essential at scale):**
14. **APM** — Application Performance Monitoring (Datadog, New Relic)
15. **Health checks** — `/health` endpoint for load balancer
16. **Alerting** — Alert on error rate spikes, high latency, memory growth

### Q4: Compare session-based authentication vs JWT. When would you use each?

**Model Answer:**

| Aspect | Session-based | JWT |
|--------|-------------|-----|
| **State** | Stateful (session stored server-side) | Stateless (token contains claims) |
| **Storage** | Server: memory/Redis/DB. Client: session ID cookie | Client: httpOnly cookie or localStorage |
| **Scaling** | Needs shared session store (Redis) across servers | Scales easily — any server can verify |
| **Revocation** | Easy — delete session from store | Hard — need blocklist or short expiry |
| **Payload size** | Small (session ID cookie ~32 bytes) | Larger (token ~800+ bytes) |
| **Security** | No sensitive data exposed | Payload is base64-encoded (readable!), not encrypted |

**Use sessions when:**
- Traditional server-rendered web apps
- Need immediate revocation (logout everywhere)
- Already have Redis for caching

**Use JWT when:**
- API-first / microservices architecture
- Distributed systems without shared state
- Mobile app backends
- Short-lived operations (e.g., email verification links)

**Best practice for SPAs:** Use JWT with refresh token rotation, stored in httpOnly secure cookies (NOT localStorage, which is vulnerable to XSS).

### Q5: How would you test a Node.js REST API end-to-end?

**Model Answer:**

**Test Setup:**
```javascript
// Use in-memory DB for isolation
beforeAll(async () => {
    mongo = await MongoMemoryServer.create();
    await mongoose.connect(mongo.getUri());
});
afterEach(() => User.deleteMany({})); // Clean state
afterAll(() => { mongoose.disconnect(); mongo.stop(); });
```

**Test Categories:**

1. **Happy path** — Valid inputs produce expected outputs
   ```javascript
   test('POST /users creates user', async () => {
       const res = await request(app)
           .post('/api/users').send({ name: 'Alice', email: 'a@b.com' })
           .expect(201);
       expect(res.body.data).toHaveProperty('id');
   });
   ```

2. **Validation errors** — Invalid inputs return 400

3. **Authentication** — Protected routes return 401 without token

4. **Authorization** — Non-admin users get 403 on admin routes

5. **Not found** — Non-existent resources return 404

6. **Edge cases** — Empty arrays, duplicate entries (409), very long inputs

7. **Error scenarios** — DB connection failure, timeout handling

**Coverage:** Aim for 80%+ on business logic, 60%+ overall. Use `--coverage` flag.

### Q6: When would you choose microservices over a monolith for a Node.js application?

**Model Answer:**

**Start with a monolith when:**
- Small team (< 10 developers)
- New product / MVP (requirements unclear)
- Simple domain (e.g., CRUD API)
- Speed of development is the priority

**Consider microservices when:**
- Team grows beyond 10+ developers working on different features
- Parts of the app need independent scaling (e.g., image processing vs. user auth)
- Different components benefit from different technologies
- Deployment independence is needed (deploy payments without risking notifications)
- The monolith has become too large to understand, test, and deploy safely

**The cost of microservices:**
- Network complexity (service discovery, retries, circuit breakers)
- Data consistency challenges (no simple transactions across services)
- Operational overhead (monitoring, logging, tracing across services)
- E2E testing becomes harder

**The middle ground: "Modular monolith"** — structure your monolith with clear module boundaries (separate folders/packages per domain), so you CAN split into microservices later when the need arises. Don't start with microservices to solve a problem you don't have yet.

> **Key interview phrase:** "Microservices trade development simplicity for operational complexity. Only make that trade when the benefits outweigh the costs for your team and product."